# Lightweight Self-Supervised Text Representation Learning
## Google Colab / Kaggle End-to-End Training & Evaluation Launcher

**Project:** A Lightweight Self-Supervised Framework for Text Representation Learning Using Masked Latent Prediction.

This executable notebook:
1. Pins to an explicit verified Git commit.
2. Installs the tested environment and verifies the deterministic unit test suite.
3. Loads real WikiText pilot data via `src.data.get_dataset_loaders` enforcing `docs/INTERFACES.md`.
4. Instantiates the small scratch encoder ($L=128, D=256$, 4 layers, 4 heads).
5. Executes an **actual pilot training run** with real loss logging.
6. Saves a resumable checkpoint containing the `student` key and `model_config`.
7. Reloads and freezes the checkpoint using `src.evaluate.load_student_checkpoint`.
8. Measures representation geometry (RankMe effective rank, dimension variance, centered cosine distribution) on held-out validation data.
9. Evaluates downstream transfer with a frozen linear probe on AG News.

> **Security & Auth Notice:** Never embed GitHub personal access tokens or credentials in this notebook. In Colab, store private keys in Colab Secrets (`userdata.get('GH_TOKEN')`) or enter credentials interactively.

### 1. Hardware & Environment Inspection

In [ ]:
!nvidia-smi

import os
import sys
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__}")
print(f"Compute Device:  {device}")
if torch.cuda.is_available():
    print(f"GPU Name:        {torch.cuda.get_device_name(0)}")
    print(f"VRAM Allocated:  {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

### 2. Git Setup & Checkout to Pinned Commit
We checkout an explicit, tested commit hash to guarantee experimental reproducibility across runtime restarts.

In [ ]:
# Pin to the explicit verified Person 2 milestone commit
PINNED_COMMIT = "2085aad"

# If running inside Google Colab without cloned repository:
if not os.path.exists("src"): 
    print("Cloning repository...")
    !git clone https://github.com/dakshadev/7th-sem.git .

!git fetch origin person2
!git checkout {PINNED_COMMIT}
!git log -n 1 --oneline

### 3. Dependency Installation & Test Suite Verification
Install tested dependency versions and verify all CPU unit tests pass.

In [ ]:
!pip install -q -r requirements.txt
!python -m pytest tests/ -v

### 4. Real Dataset Loader Entry Point (`src.data`)
Load real WikiText-2 pilot data (`wikitext-2-raw-v1`) using `get_dataset_loaders`.
Note on labelling: WikiText-2 is the pilot corpus (~2M tokens raw) used for rapid prototyping and validation; WikiText-103 (`wikitext-103-raw-v1`, ~5M token subset) is the primary pretraining target.

In [ ]:
from src.data import get_dataset_loaders, get_bert_tokenizer, count_tokens_in_batch, decode_masked_example

tokenizer = get_bert_tokenizer()
train_loader, val_loader, data_manifest = get_dataset_loaders(
    dataset_config="wikitext-2-raw-v1",
    batch_size=8,
    max_length=128,
    mask_ratio=0.15,
    mask_mode="independent",
    seed=42,
    max_train_samples=500,  # Pilot subset for quick execution
    max_val_samples=100,
    tokenizer=tokenizer,
)

print("Data Manifest:", data_manifest)
sample_batch = next(iter(train_loader))
print("\nBatch Contract Tensors Verified:")
for k, v in sample_batch.items():
    print(f"  {k:18s} shape={list(v.shape)} dtype={v.dtype}")

counts = count_tokens_in_batch(sample_batch)
print(f"Tokens in batch: {counts}")

### 5. Scratch Small Text Encoder & MLM Head
Implemented from scratch matching `docs/PROJECT_CONTEXT.md` specifications:
- Sequence length $L=128$, hidden dimension $D=256$, 4 encoder layers, 4 attention heads, FFN dimension 1024.
- Randomly initialized learned token and position embeddings (no pretrained encoder weights!).
- Vocabulary projection head (`256 -> 30522`).

In [ ]:
import torch.nn as nn

class SmallScratchTextEncoder(nn.Module):
    def __init__(self, vocab_size=30522, hidden_dim=256, num_layers=4, num_heads=4, max_len=128, dropout=0.1):
        super().__init__()
        self.model_config = {
            "vocab_size": vocab_size,
            "hidden_dim": hidden_dim,
            "num_layers": num_layers,
            "num_heads": num_heads,
            "max_len": max_len,
            "dropout": dropout,
        }
        self.token_embed = nn.Embedding(vocab_size, hidden_dim)
        self.pos_embed = nn.Embedding(max_len, hidden_dim)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=1024,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.mlm_head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_ids, attention_mask=None):
        B, L = input_ids.shape
        pos_ids = torch.arange(L, device=input_ids.device).unsqueeze(0).expand(B, L)
        x = self.layer_norm(self.token_embed(input_ids) + self.pos_embed(pos_ids))
        
        # PyTorch attention mask expects True for positions to be masked (padding)
        key_padding_mask = ~attention_mask if attention_mask is not None else None
        h = self.transformer(x, src_key_padding_mask=key_padding_mask)
        logits = self.mlm_head(h)
        return h, logits

model = SmallScratchTextEncoder().to(device)
print(f"Encoder initialized with {sum(p.numel() for p in model.parameters()):,} parameters.")

### 6. Actual Pilot Pretraining Run
Execute real optimization steps on the WikiText pilot data, computing cross-entropy loss strictly on `prediction_mask` positions.

In [ ]:
import time
import matplotlib.pyplot as plt

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

num_steps = 60
losses = []
total_nonpad_tokens = 0
total_targets = 0

model.train()
start_time = time.time()
step = 0

while step < num_steps:
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        masked_input_ids = batch["masked_input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        prediction_mask = batch["prediction_mask"].to(device)

        if not prediction_mask.any():
            continue

        optimizer.zero_grad()
        hidden_states, logits = model(masked_input_ids, attention_mask)

        # Cross entropy strictly at eligible masked prediction positions
        active_logits = logits[prediction_mask]
        active_targets = input_ids[prediction_mask]
        loss = criterion(active_logits, active_targets)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        losses.append(loss.item())
        total_nonpad_tokens += int(attention_mask.sum().item())
        total_targets += int(prediction_mask.sum().item())
        step += 1

        if step % 15 == 0 or step == num_steps:
            elapsed = time.time() - start_time
            print(f"Step {step:3d}/{num_steps} | Loss: {loss.item():.4f} | "
                  f"Tokens: {total_nonpad_tokens} | Targets: {total_targets} | Time: {elapsed:.2f}s")
        if step >= num_steps:
            break

# Plot the real measured training loss curve
plt.figure(figsize=(8, 4))
plt.plot(losses, color="#1f77b4", linewidth=2, label="MLM Cross-Entropy Loss")
plt.title("Pilot Pretraining Loss Curve (Real WikiText-2 Pilot)", fontsize=13, fontweight="bold")
plt.xlabel("Optimization Step", fontsize=11)
plt.ylabel("Loss", fontsize=11)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

### 7. Save & Reload Resumable Checkpoint
Save checkpoint using the verified contract schema (containing `student` key and `model_config`).
Reload and freeze the student encoder using `src.evaluate.load_student_checkpoint`.

In [ ]:
from src.evaluate import load_student_checkpoint

checkpoint_payload = {
    "student": model.state_dict(),
    "model_config": model.model_config,
    "step": step,
    "optimizer": optimizer.state_dict(),
}

ckpt_path = "results/pilot_training_checkpoint.pt"
torch.save(checkpoint_payload, ckpt_path)
print(f"Saved training checkpoint to {ckpt_path}")

# Instantiate fresh model and reload using 'student' key & model_config
eval_model = SmallScratchTextEncoder().to(device)
frozen_student, recovered_cfg = load_student_checkpoint(ckpt_path, eval_model, return_config=True)

print("\nCheckpoint successfully reloaded!")
print(f"Recovered Config: {recovered_cfg}")
print(f"Model is frozen:  {all(not p.requires_grad for p in frozen_student.parameters())}")
print(f"Eval mode active: {not frozen_student.training}")

### 8. Held-Out Representation Geometry Diagnostics
Evaluate the frozen student encoder on held-out validation sequences:
- Uncorrupted input text is passed to the student encoder.
- Mean pooling strictly excludes padding and special tokens (`[CLS]`, `[SEP]`).
- Computes centered effective rank (RankMe singular value entropy and covariance eigenvalue entropy), dimension variance, and centered pairwise cosine distribution.

In [ ]:
from src.evaluate import masked_mean_pooling, compute_representation_geometry

special_ids = set(tokenizer.all_special_ids)
pooled_embeddings = []

frozen_student.eval()
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        # Evaluation contract: student receives UNCORRUPTED input_ids
        hidden_states, _ = frozen_student(input_ids, attention_mask)
        pooled = masked_mean_pooling(hidden_states, attention_mask, input_ids, special_ids)
        pooled_embeddings.append(pooled.cpu())

Z_val = torch.cat(pooled_embeddings, dim=0).numpy()
geometry_results = compute_representation_geometry(Z_val)

print("\n--- Held-Out Validation Geometry Diagnostics ---")
print(f"Status:                      {geometry_results['status']} (Degenerate: {geometry_results['is_degenerate']})")
print(f"Evaluated Samples N:         {geometry_results['num_samples']}")
print(f"Embedding Dimension D:       {geometry_results['dimension']}")
print(f"Effective Rank (RankMe SV):  {geometry_results['effective_rank_singular']:.2f} / {min(geometry_results['num_samples'], geometry_results['dimension'])}")
print(f"Effective Rank (Covariance): {geometry_results['effective_rank_covariance']:.2f}")
print(f"Mean Dimension Variance:     {geometry_results['mean_dimension_variance']:.4f}")
print(f"Dead Dimensions Count:       {geometry_results['dead_dimension_count']} ({geometry_results['dead_dimension_fraction']*100:.1f}%)")
print(f"Mean Pairwise Cosine:        {geometry_results['mean_pairwise_cosine']:.4f} (std: {geometry_results['std_pairwise_cosine']:.4f})")

### 9. Downstream Transfer: Frozen Linear Probe on AG News
Extract pooled uncorrupted representations from AG News training and testing splits.
Fit a `StandardScaler` strictly on training representations, train logistic regression, and evaluate test accuracy and macro-F1.

In [ ]:
from datasets import load_dataset
from src.evaluate import evaluate_frozen_linear_probe

# Load small sample of AG News
ag_train = load_dataset("fancyzhx/ag_news", split="train[:200]")
ag_test = load_dataset("fancyzhx/ag_news", split="test[:100]")

def extract_features(dataset):
    features = []
    labels = []
    with torch.no_grad():
        for ex in dataset:
            enc = tokenizer(ex["text"], max_length=128, truncation=True, padding="max_length", return_tensors="pt")
            inp = enc["input_ids"].to(device)
            att = enc["attention_mask"].bool().to(device)
            h, _ = frozen_student(inp, att)
            p = masked_mean_pooling(h, att, inp, special_ids)
            features.append(p.squeeze(0).cpu().numpy())
            labels.append(ex["label"])
    return np.array(features), np.array(labels)

X_train, y_train = extract_features(ag_train)
X_test, y_test = extract_features(ag_test)

probe_results = evaluate_frozen_linear_probe(X_train, y_train, X_test, y_test, seed=42)

print("\n--- AG News Frozen Linear Probe Results ---")
print(f"Held-Out Accuracy:    {probe_results['accuracy']:.4f}")
print(f"Macro-F1 Score:       {probe_results['macro_f1']:.4f}")
print(f"Majority Baseline:    {probe_results['majority_baseline']:.4f}")
print(f"Training Samples:     {probe_results['train_samples']}")
print(f"Test Samples:         {probe_results['test_samples']}")